# FinMamba3 LOB Pretraining (Polymarket / FI-2010, Mamba3 SISO or MIMO)

Pretraining the Mamba3 world model on Polymarket binary-outcome LOBs (default) or the FI-2010 Nasdaq Helsinki benchmark (Ntakaris et al. 2018). Set `HF_TOKEN` as an environment variable before launching Jupyter (e.g. `export HF_TOKEN=hf_...`) and then run all cells.

SISO works on any CUDA GPU. This branch defaults to `USE_MIMO = True`, the richer cross-channel MIMO scan, which requires the TileLang kernel and has no Python fallback. For the full `d_state=128` MIMO comparison, use H100/H200 (sm_90) or B200 (sm_100). A100 can fit the compatibility `chunk_size=8` path, but the observed Polymarket MIMO run was ~9.49 s/it (~39.5 hours for 15k steps), so it is not practical for the final MIMO-vs-SISO number. L4/T4 and workstation Blackwell cards generally need `USE_MIMO = False` or a non-comparable `MIMO_D_STATE = 64` run.

Switch `DATASET` in the first cell between `"polymarket"` (default) and `"fi2010"`.

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Ruuudy1/FinMamba3.git"
BRANCH = "external-h100"

# Dataset selector: "fi2010" pulls the Ntakaris+2018 NoAuction DecPre CF files
# and trains on a public LOB benchmark. "polymarket" reproduces the original
# SQLite pipeline. Both share the same Mamba3 world-model code.
DATASET = "polymarket"
assert DATASET in ("polymarket", "fi2010"), DATASET

WORK_ROOT = Path.home() / 'finmamba3_workspace'
PROJECT_DIR = str(WORK_ROOT / 'Drama')
CACHE_ROOT = WORK_ROOT / 'finmamba3_cache'
DATA_ROOT = WORK_ROOT / 'finmamba3_data'
WORK_ROOT.mkdir(parents=True, exist_ok=True)

# Data lives in the dataset repo (data/ + logs/ only). Checkpoints and prebuilt
# CUDA wheels now live in dedicated repos so the dataset repo stays data-only.
HF_REPO = "sj-hryi/FinMamba3"
CKPT_REPO = "sj-hryi/FinMamba3-checkpoints"
WHEELS_REPO = "sj-hryi/FinMamba3-wheels"
HF_REVISION = None
FORCE_REBUILD_WHEELS = False

# Default "Run All" path: full pretraining on the active dataset.
# Flip RUN_PROBES to True to run the 3-probe collapse-diagnosis sweep instead.
# SMOKE_TEST clips both probes and pretrain to 20 steps for plumbing verification.
SMOKE_TEST = False
RUN_PROBES = False
HOURS_TRAIN = 6
HOURS_VAL = 1
PROBE_STEPS = 20 if SMOKE_TEST else 1000
MAX_STEPS = 20 if SMOKE_TEST else 15000

# MIMO is on by default on this branch. It requires the TileLang kernel with no
# Python fallback. RegimeFiLM is off to compare apples-to-apples against the known
# SISO baseline (val ~368, also FiLM-off). Flip USE_REGIME_FILM=True for best-of-both.
USE_MIMO = True
USE_REGIME_FILM = False

# The TileLang MIMO kernel hard-requires chunk_size >= 8 and at its default
# chunk_size=16 needs ~219 KB of dynamic shared memory, which only H100-class
# cards (~227 KB/SM) have. Chunk_size is only a tiling parameter so the trained
# model is identical regardless of which value is used.
MIMO_CHUNK_SIZE = 8

# Lowering d_state (default 128) also cuts MIMO kernel SMEM as an escape hatch
# for ~100 KB cards. d_state=64 changes the model so pair it with a matched SISO
# baseline, not the d_state=128 val-368 number. None keeps the default 128.
MIMO_D_STATE = None

CONFIG_FILENAME_BY_DATASET = {
    "polymarket": "lob.yaml",
    "fi2010": "fi2010.yaml",
}
CONFIG_FILENAME = CONFIG_FILENAME_BY_DATASET[DATASET]

def pip_install(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

def get_hf_token():
    return os.environ.get('HF_TOKEN')

print(f"Project at {PROJECT_DIR}")
print(f"Dataset: {DATASET}, config: {CONFIG_FILENAME}")
print(f"Mode: {'SMOKE_TEST' if SMOKE_TEST else 'probes' if RUN_PROBES else 'full pretrain'} | MAX_STEPS={MAX_STEPS}")
print(f"Arch: {'MIMO' if USE_MIMO else 'SISO'} | RegimeFiLM: {USE_REGIME_FILM} | MIMO_CHUNK_SIZE: {MIMO_CHUNK_SIZE} | MIMO_D_STATE: {MIMO_D_STATE}")


In [ ]:
HF_TOKEN = get_hf_token()
if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN missing. Set the HF_TOKEN environment variable before launching Jupyter (e.g. export HF_TOKEN=hf_... in your shell)."
    )

pip_install('huggingface_hub')
from huggingface_hub import snapshot_download

DATA_ROOT.mkdir(parents=True, exist_ok=True)

if DATASET == "polymarket":
    # Bundles live under data/polymarket/{train,validation,test}.tar.gz on the HF
    # mirror. Each .tar.gz holds polymarket.db + polymarket_books/ + binance_lob/.
    # Training uses train + validation; the test split is kept on HF for separate
    # evaluation and is not pulled here.
    print('Pulling Polymarket dataset bundle from HuggingFace Hub.')
    snapshot_download(
        repo_id=HF_REPO, repo_type='dataset',
        allow_patterns=['data/polymarket/train.tar.gz', 'data/polymarket/validation.tar.gz'],
        revision=HF_REVISION, token=HF_TOKEN,
        local_dir=str(DATA_ROOT),
    )
    TRAIN_GZ = DATA_ROOT / 'data' / 'polymarket' / 'train.tar.gz'
    VAL_GZ = DATA_ROOT / 'data' / 'polymarket' / 'validation.tar.gz'
    for p in (TRAIN_GZ, VAL_GZ):
        if not p.exists():
            raise FileNotFoundError(p)
    print('Polymarket bundles ready:', TRAIN_GZ.name, VAL_GZ.name)
elif DATASET == "fi2010":
    # FI-2010 NoAuction DecPre CF files (Ntakaris+2018), 3-way split on the HF
    # mirror: train/ (85% of the original train file), validation/ (carved last
    # 15%), test/ (the original Test_ file). Training uses train + validation;
    # the test split lives at data/fi2010/test/ on HF and is not pulled here.
    print('Pulling FI-2010 NoAuction DecPre CF files from HuggingFace Hub.')
    snapshot_download(
        repo_id=HF_REPO, repo_type='dataset',
        allow_patterns=[
            'data/fi2010/train/Train_Dst_NoAuction_DecPre_CF_7.txt',
            'data/fi2010/validation/Val_Dst_NoAuction_DecPre_CF_7.txt',
        ],
        revision=HF_REVISION, token=HF_TOKEN,
        local_dir=str(DATA_ROOT),
    )
    FI2010_TRAIN = DATA_ROOT / 'data' / 'fi2010' / 'train' / 'Train_Dst_NoAuction_DecPre_CF_7.txt'
    FI2010_VAL = DATA_ROOT / 'data' / 'fi2010' / 'validation' / 'Val_Dst_NoAuction_DecPre_CF_7.txt'
    for p in (FI2010_TRAIN, FI2010_VAL):
        if not p.exists():
            raise FileNotFoundError(
                f"FI-2010 file missing at {p}. The mirror lives at "
                "sj-hryi/FinMamba3 under data/fi2010/{train,validation,test}/."
            )
    print('FI-2010 files ready:', FI2010_TRAIN.name, FI2010_VAL.name)
else:
    raise ValueError(f"Unknown DATASET: {DATASET!r}")


In [ ]:
import shutil

if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)
subprocess.check_call(['git', 'clone', '--branch', BRANCH, REPO_URL, PROJECT_DIR])
os.chdir(PROJECT_DIR)
print('Repo ready:', os.getcwd())


In [ ]:
os.chdir(PROJECT_DIR)

pip_cache = CACHE_ROOT / 'pip'
pip_cache.mkdir(parents=True, exist_ok=True)
os.environ['PIP_CACHE_DIR'] = str(pip_cache)

pip_install('--upgrade', 'pip')
pip_install('huggingface_hub[cli]')
pip_install('packaging', 'ninja', 'setuptools==69.5.1', 'numpy>=2,<3')

# Older GPU rental images ship typing_extensions < 4.10.0, which lacks TypeIs.
# Unconditional shim is safe because torch uses TypeIs for type annotations only,
# not runtime logic. TypeGuard is the closest substitute and always available on
# Python 3.11+ images.
import typing_extensions as _typing_ext
from typing_extensions import TypeGuard as _TypeIs_shim
_typing_ext.TypeIs = _TypeIs_shim
del _typing_ext, _TypeIs_shim

# H100 NVL (sm_90) is supported by torch 2.7/cu126. torch 2.7 also switched libtorch
# to _GLIBCXX_USE_CXX11_ABI=1 on Linux pip wheels, matching the prebuilt causal-conv1d
# and mamba-ssm wheels (which are all cxx11=1 despite the FALSE label in their filenames).
pip_install('torch==2.7.0', 'torchvision==0.22.0', 'torchaudio==2.7.0',
            '--index-url', 'https://download.pytorch.org/whl/cu126')

import torch
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability(0)
else:
    major, minor = 9, 0
arch_list = f"{major}.{minor}"
os.environ['TORCH_CUDA_ARCH_LIST'] = arch_list
print(f"TORCH_CUDA_ARCH_LIST: {arch_list}")

# Install causal-conv1d and mamba-ssm from their official GitHub release wheels.
# The release filename encodes the exact (cuda, torch, cxx11_abi, python) tuple,
# so picking the wheel matching torch's ABI avoids the libtorch symbol mismatch
# that source builds hit when wheel ABI does not equal torch's _GLIBCXX_USE_CXX11_ABI.
cxx11_abi = 'TRUE' if torch._C._GLIBCXX_USE_CXX11_ABI else 'FALSE'
torch_minor = '.'.join(torch.__version__.split('+')[0].split('.')[:2])
cuda_major = (torch.version.cuda or '12.4').split('.')[0]
py_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"

CAUSAL_CONV1D_VERSION = '1.6.2.post1'
MAMBA_SSM_VERSION = '2.3.2.post1'
causal_conv1d_url = (
    f"https://github.com/Dao-AILab/causal-conv1d/releases/download/v{CAUSAL_CONV1D_VERSION}/"
    f"causal_conv1d-{CAUSAL_CONV1D_VERSION}+cu{cuda_major}torch{torch_minor}cxx11abi{cxx11_abi}-{py_tag}-{py_tag}-linux_x86_64.whl"
)
mamba_ssm_url = (
    f"https://github.com/state-spaces/mamba/releases/download/v{MAMBA_SSM_VERSION}/"
    f"mamba_ssm-{MAMBA_SSM_VERSION}+cu{cuda_major}torch{torch_minor}cxx11abi{cxx11_abi}-{py_tag}-{py_tag}-linux_x86_64.whl"
)
print(f"Installing causal-conv1d prebuilt: cu{cuda_major}/torch{torch_minor}/abi{cxx11_abi}/{py_tag}")
pip_install('--force-reinstall', '--no-deps', causal_conv1d_url)
print(f"Installing mamba-ssm prebuilt: cu{cuda_major}/torch{torch_minor}/abi{cxx11_abi}/{py_tag}")
pip_install('--force-reinstall', '--no-deps', mamba_ssm_url)
pip_install('-e', '.')
pip_install('tilelang==0.1.8')
# tilelang 0.1.8 pulls apache-tvm-ffi 0.1.11, but the bundled TVM Python
# bindings (NestedLoopChecker visitor) crash with that version because
# TVMDerivedObject.__setattr__ reads self._inst before __init__ has set it.
# Pinning to 0.1.9 matches mamba-ssm's declared upper bound and restores the
# working visitor base class.
pip_install('apache-tvm-ffi==0.1.9', '--force-reinstall', '--no-deps')
pip_install('quack-kernels')
pip_install('transformers')
# Rental GPU networks occasionally drop mid-transfer on this ~200 MB package.
for _attempt in range(5):
    _result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--timeout', '300', 'triton>=3.5.0'])
    if _result.returncode == 0:
        break
if _result.returncode:
    raise subprocess.CalledProcessError(_result.returncode, _result.args)

import torch, numpy as np
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.version.cuda)
print('numpy', np.__version__)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

# Some torch builds ship pytorch-triton without set_allocator; mamba_ssm requires
# the symbol to exist at import time.
import triton
if 'set_allocator' not in triton.__dict__:
    triton.set_allocator = lambda fn: None

import causal_conv1d_cuda
from mamba_ssm.modules.mamba3 import Mamba3
print('Mamba3 imported:', Mamba3)
from mamba_ssm.ops.tilelang.mamba3.mamba3_mimo import mamba3_mimo
print('Mamba3 MIMO TileLang import: ok', mamba3_mimo)


In [ ]:
import shutil, tarfile

project = Path(PROJECT_DIR)
data_dir = project / 'data'
train_dir = data_dir / 'train'
val_dir = data_dir / 'validation'


def _is_metadata_path(path):
    parts = [str(x) for x in Path(path).parts]
    return any(part == '__MACOSX' or part.startswith('._') for part in parts)


def _extract_targz(archive, out):
    out.mkdir(parents=True, exist_ok=True)
    print('Extracting', archive, '->', out)
    with tarfile.open(archive, 'r:gz') as tf:
        members = [m for m in tf.getmembers() if not _is_metadata_path(m.name)]
        if sys.version_info >= (3, 12):
            tf.extractall(out, members=members, filter='data')
        else:
            tf.extractall(out, members=members)


if DATASET == "polymarket":
    if (train_dir / 'polymarket.db').exists() and (val_dir / 'polymarket.db').exists():
        print('Polymarket data already prepared at', data_dir)
    else:
        for split, gz in (('train', TRAIN_GZ), ('validation', VAL_GZ)):
            out = data_dir / split
            shutil.rmtree(out, ignore_errors=True)
            _extract_targz(gz, out)
        print('Prepared train:', train_dir)
        print('Prepared validation:', val_dir)
elif DATASET == "fi2010":
    src_train = DATA_ROOT / 'data' / 'fi2010' / 'train' / 'Train_Dst_NoAuction_DecPre_CF_7.txt'
    src_val = DATA_ROOT / 'data' / 'fi2010' / 'validation' / 'Val_Dst_NoAuction_DecPre_CF_7.txt'
    train_dir.mkdir(parents=True, exist_ok=True)
    val_dir.mkdir(parents=True, exist_ok=True)
    dst_train = train_dir / src_train.name
    dst_val = val_dir / src_val.name
    if not dst_train.exists():
        shutil.copy(src_train, dst_train)
    if not dst_val.exists():
        shutil.copy(src_val, dst_val)
    print('Prepared FI-2010 train:', dst_train)
    print('Prepared FI-2010 validation:', dst_val)
else:
    raise ValueError(f"Unknown DATASET: {DATASET!r}")


In [ ]:
import re, datetime
from pathlib import Path

os.chdir(PROJECT_DIR)

# Each probe flips one hardcoded constant from default to test a collapse hypothesis.
# H1 raises the representation KL weight to match DreamerV3 0.5/0.5 symmetry.
# H2 zeros the free-bits floor to free the dynamics signal from the 1-nat clip.
# H3 lowers the decoder size weight so price features can compete with depth features.
PROBES = [
    ("H1_rep_loss_weight_0p5", ['--Models.WorldModel.RepresentationLossWeight', '0.5']),
    ("H2_free_bits_0p0",       ['--Models.WorldModel.FreeBits', '0.0']),
    ("H3_size_weight_1p0",     ['--Models.WorldModel.Decoder.SizeWeight', '1.0']),
]

RUN_DATE = datetime.datetime.now(datetime.UTC).strftime('%Y%m%d_%H%M%S')
LOG_DIR = WORK_ROOT / 'probe_logs' / RUN_DATE
LOG_DIR.mkdir(parents=True, exist_ok=True)
STDOUT_PATH = LOG_DIR / 'stdout.txt'

# Every run's stdout is appended to one shared log file. Tqdm progress bars
# are thinned to GPU-util cadence to keep the file small.
_log_handle = STDOUT_PATH.open('w', encoding='utf-8')
_TQDM_RE = re.compile(r'^pretrain:\s*\d+%\|.*\|\s*(\d+)/(\d+)\s*\[')
_GPU_RE = re.compile(r'\[GPU\] util=')
_state = {'gpu_seen_since_last_progress': True}


def _log_filtered(line):
    print(line, end='')
    if _GPU_RE.search(line):
        _state['gpu_seen_since_last_progress'] = True
    m = _TQDM_RE.match(line)
    if m:
        step, total = int(m.group(1)), int(m.group(2))
        is_endpoint = step <= 1 or step == total
        if is_endpoint or _state['gpu_seen_since_last_progress']:
            _log_handle.write(line)
            _state['gpu_seen_since_last_progress'] = False
        return
    _log_handle.write(line)
    _log_handle.flush()


CONFIG_PATH = Path(PROJECT_DIR) / 'configs' / CONFIG_FILENAME


def run_train(steps, extra_args, label):
    _log_filtered(f"\n{'='*72}\n{label}\n{'='*72}\n")
    cmd = [
        sys.executable, '-u', '-B', '-m', 'finmamba3.train',
        '--config', str(CONFIG_PATH),
        '--dataset', DATASET,
        '--hours-train', str(HOURS_TRAIN),
        '--hours-val', str(HOURS_VAL),
        '--JointTrainAgent.SampleMaxSteps', str(steps),
        *extra_args,
    ]
    _log_filtered('Running: ' + ' '.join(cmd) + '\n')
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        _log_filtered(line)
    rc = proc.wait()
    if rc:
        raise subprocess.CalledProcessError(rc, cmd)


run_names = []
if RUN_PROBES:
    for probe_name, probe_args in PROBES:
        run_names.append(probe_name)
        run_train(PROBE_STEPS, probe_args, f"Probe {probe_name} ({PROBE_STEPS} steps)")
else:
    run_names.append('full_pretrain')
    if USE_MIMO:
        from mamba_ssm.ops.tilelang.mamba3.mamba3_mimo import mamba3_mimo  # noqa: F401
        gpu_cap = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
        d_state = MIMO_D_STATE if MIMO_D_STATE is not None else 128
        # H100/H200 (sm_90), B200 (sm_100), and A100 (sm_80) all fit chunk_size=8.
        # Only RTX/L4/workstation-Blackwell (~100 KB) cards need d_state<=64.
        if gpu_cap not in [(9, 0), (10, 0), (8, 0)]:
            if gpu_cap in [(8, 6), (8, 9), (12, 0)] and d_state > 64:
                raise RuntimeError(
                    f'USE_MIMO=True on sm_{gpu_cap[0]}{gpu_cap[1]} with d_state={d_state}: '
                    f'MIMO backward needs ~120 KB of shared memory at d_state=128 but '
                    f'this card has only ~100 KB. Set MIMO_D_STATE=64 for a non-comparable '
                    f'run, switch to H100/H200/B200, or set USE_MIMO=False.'
                )
            elif gpu_cap not in [(8, 6), (8, 9), (12, 0)]:
                raise RuntimeError(
                    f'USE_MIMO=True on sm_{gpu_cap[0]}{gpu_cap[1]}: too little shared '
                    f'memory even at d_state=64. Use H100/H200/B200 or set USE_MIMO=False.'
                )
    else:
        gpu_cap = (0, 0)
    extra_args = [
        '--Models.WorldModel.Mamba3.is_mimo', 'true' if USE_MIMO else 'false',
        '--Models.WorldModel.RegimeFiLM.Enabled', 'true' if USE_REGIME_FILM else 'false',
    ]
    # H100/H200 (sm_90) and B200 (sm_100) match the kernel default chunk_size so no override.
    if USE_MIMO and gpu_cap not in [(9, 0), (10, 0)]:
        extra_args += ['--Models.WorldModel.Mamba3.chunk_size', str(MIMO_CHUNK_SIZE)]
        print(f"Non-H100-class GPU sm_{gpu_cap[0]}{gpu_cap[1]}: setting MIMO chunk_size={MIMO_CHUNK_SIZE} to fit shared memory.")
        if MIMO_D_STATE is not None:
            extra_args += ['--Models.WorldModel.Mamba3.d_state', str(MIMO_D_STATE)]
            print(f"Also overriding MIMO d_state={MIMO_D_STATE} (changes the model; pair with a matched SISO baseline).")
    arch_label = 'MIMO' if USE_MIMO else 'SISO'
    film_label = ' + RegimeFiLM' if USE_REGIME_FILM else ''
    run_train(MAX_STEPS, extra_args, f'Full Mamba3 {arch_label}{film_label} pretrain ({MAX_STEPS} steps)')
_log_handle.close()

# Recover wandb run ids from the log in chronological order. De-dup while
# preserving order; each wandb.init() prints the same id several times.
log_text = STDOUT_PATH.read_text(encoding='utf-8')
ordered_run_ids = []
for run_id in re.findall(r'offline-run-\d+_\d+-(\w+)', log_text):
    if run_id not in ordered_run_ids:
        ordered_run_ids.append(run_id)

# Copy each run's wandb-summary.json next to the stdout log.
WANDB_ROOT = Path(PROJECT_DIR) / 'wandb'
for run_name, run_id in zip(run_names, ordered_run_ids):
    summary_matches = list(WANDB_ROOT.glob(f'offline-run-*-{run_id}/files/wandb-summary.json'))
    if not summary_matches:
        print(f"warning: no wandb-summary.json for {run_name} ({run_id})")
        continue
    dst = LOG_DIR / f"{run_name}_{run_id}_summary.json"
    shutil.copy(summary_matches[0], dst)
    print(f"copied {summary_matches[0].name} -> {dst.name}")

# Upload the cell-run log dir to HF Hub under logs/, alongside checkpoints/.
hf_token = get_hf_token()
if hf_token and HF_REPO:
    from huggingface_hub import HfApi
    HfApi().upload_folder(
        folder_path=str(LOG_DIR),
        path_in_repo=f"logs/{RUN_DATE}",
        repo_id=HF_REPO, repo_type='dataset', token=hf_token,
    )
    print(f"\nUploaded run logs to https://huggingface.co/datasets/{HF_REPO}/tree/main/logs/{RUN_DATE}")
else:
    print(f"\nHF_TOKEN missing; logs only saved locally at {LOG_DIR}")


In [ ]:
# Read-only dump of every offline wandb run's full summary. Use this to recover
# Imagine/mid_norm_std and per-feature MSE values that wandb truncates with
# `+1 ...` in its CLI summary block.
import json
wandb_root = Path(PROJECT_DIR) / 'wandb'
for d in sorted(wandb_root.glob('offline-run-*')):
    p = d / 'files' / 'wandb-summary.json'
    print('==', d, '==')
    if p.exists():
        with p.open() as f:
            print(json.dumps(json.load(f), indent=2))
    else:
        print(f"(no summary file at {p})")
    print()


In [ ]:
src = Path(PROJECT_DIR) / 'saved_models' / 'lob'
if not src.exists():
    print('No checkpoint directory found yet:', src)
else:
    hf_token = get_hf_token()
    if hf_token and CKPT_REPO:
        from huggingface_hub import HfApi
        HfApi().upload_folder(
            folder_path=str(src),
            path_in_repo='checkpoints/lob',
            repo_id=CKPT_REPO, repo_type='model', token=hf_token,
        )
        print('Backed up checkpoints to HF Hub:', CKPT_REPO)
    else:
        print('HF_TOKEN not set - checkpoints only saved locally at:', src)


---
## Evaluation

The cells below run the five evaluation improvements added in the
`FinMamba3-eval-improvements` branch.  They all run **after** training
on the checkpoint produced above.  Nothing here retrains the model.

### Workflow to get these cells into Colab
1. **Unzip** `FinMamba3-eval-improvements.zip` (the file you downloaded).
2. **Push** the changes to your GitHub fork of FinMamba3.
3. Make sure `REPO_URL` and `BRANCH` in the first cell point at your fork and branch.
4. Re-run the notebook from the top — the git-clone cell will pull the new code.

If you only want to run eval on an existing checkpoint without re-training,
run the **git-clone**, **pip-install**, and **data-extraction** cells first,
then jump straight to the eval cells below.

In [ ]:
# ── Evaluation controls ──────────────────────────────────────────────────────
# SKIP_EVAL=True skips all evaluation cells (useful when re-running training only).
# EVAL_SMOKE=True uses tiny windows for a fast plumbing check (~30 s).
# INSTALL_SKLEARN=True installs scikit-learn for the linear-probe cell.
SKIP_EVAL   = False
EVAL_SMOKE  = SMOKE_TEST   # inherit from training config; override here if needed
INSTALL_SKLEARN = True

# RUN_DATE is defined by the training cell.  Guard against running eval
# standalone (i.e. without having run the training cell first).
import datetime as _dt
if 'RUN_DATE' not in dir():
    RUN_DATE = _dt.datetime.now(_dt.timezone.utc).strftime('%Y%m%d_%H%M%S')
    print(f'RUN_DATE set to {RUN_DATE} (standalone eval mode)')


In [ ]:
import json as _json
from pathlib import Path

os.chdir(PROJECT_DIR)

ckpt_root = Path(PROJECT_DIR) / 'saved_models' / 'lob' / 'LOB'
norm_path = Path(PROJECT_DIR) / 'saved_models' / 'lob' / 'normalization.json'
config_path_str = str(Path(PROJECT_DIR) / 'configs' / CONFIG_FILENAME)
train_data_str  = str(Path(PROJECT_DIR) / 'data' / 'train')
val_data_str    = str(Path(PROJECT_DIR) / 'data' / 'validation')
eval_out_root   = Path(PROJECT_DIR) / 'eval_outputs'
eval_out_root.mkdir(parents=True, exist_ok=True)

best_ckpt = None
if ckpt_root.exists():
    # Prefer world_model_best.pth; fall back to world_model_final.pth.
    candidates = sorted(ckpt_root.glob('*/ckpt/world_model_best.pth'))
    if not candidates:
        candidates = sorted(ckpt_root.glob('*/ckpt/world_model_final.pth'))
    if candidates:
        best_ckpt = str(candidates[-1])  # most-recent run
        print(f'Checkpoint: {best_ckpt}')
        if norm_path.exists():
            print(f'Norm stats: {norm_path}')
        else:
            print('WARNING: normalization.json not found — eval scripts may fail.')
    else:
        print('No checkpoint found. Run the training cell first.')
else:
    print('No saved_models directory found. Run the training cell first.')


In [ ]:
# Diagnose whether the world model has collapsed (prior entropy near uniform,
# imagination std = 0) and which features are hardest to reconstruct.
if not SKIP_EVAL and best_ckpt:
    diag_out = eval_out_root / 'diagnose'
    diag_out.mkdir(parents=True, exist_ok=True)
    _hv = '0.1' if EVAL_SMOKE else str(HOURS_VAL)
    _cmd = [
        sys.executable, '-m', 'finmamba3.eval.diagnose_collapse',
        '--checkpoint', best_ckpt,
        '--config',     config_path_str,
        '--data-val',   val_data_str,
        '--norm-path',  str(norm_path),
        '--out-dir',    str(diag_out),
        '--hours-val',  _hv,
    ]
    subprocess.run(_cmd, check=False)
    print()
    for _p in sorted(diag_out.glob('diagnose_summary_*.json')):
        _d = _json.loads(_p.read_text())
        _r = _d['rollout']
        _e = _d['entropy']
        print(f'=== {_p.name} ===')
        print(f"  mid rollout std : {_r['mid_norm_std']:.4f}")
        print(f"  (0 = collapsed, >0.01 = alive; uniform entropy = {_e['uniform_entropy']:.3f})")
        print(f"  posterior entropy : {_e['post_entropy_mean']:.4f} "
              f"(+/- {_e['post_entropy_std']:.4f})")
        print(f"  prior entropy     : {_e['prior_entropy_mean']:.4f} "
              f"(+/- {_e['prior_entropy_std']:.4f})")
        print('  Top reconstruction errors:')
        for _item in _d['top_feature_mse'][:5]:
            print(f"    {_item['feature']:<38} mse={_item['mse']:.4f}")
        print()
elif SKIP_EVAL:
    print('Skipped (SKIP_EVAL=True)')
else:
    print('No checkpoint — skipping collapse diagnosis.')


In [ ]:
# Compare next-tick direction accuracy across the world model, LinearAR baseline,
# and (for Polymarket) a freshly trained DeepLOB.  Reports accuracy + Brier score
# at three flat-bucket thresholds so you can see the threshold sensitivity.
if not SKIP_EVAL and best_ckpt:
    _ht = '0.5' if EVAL_SMOKE else str(HOURS_TRAIN)
    _hv = '0.1' if EVAL_SMOKE else str(HOURS_VAL)
    _dir_out = eval_out_root / 'direction_comparison.md'
    _baselines = 'world_model,linear_ar'
    # DeepLOB training is slow; skip it in smoke-test mode.
    if not EVAL_SMOKE and DATASET == 'polymarket':
        _baselines = 'world_model,deeplob,linear_ar'
    _cmd = [
        sys.executable, '-m', 'finmamba3.eval.compare_direction',
        '--world-checkpoint', best_ckpt,
        '--config',      config_path_str,
        '--data-train',  train_data_str,
        '--data-val',    val_data_str,
        '--norm-path',   str(norm_path),
        '--thresholds',  '0.001,0.005,0.01',
        '--baselines',   _baselines,
        '--out',         str(_dir_out),
        '--hours-train', _ht,
        '--hours-val',   _hv,
    ]
    subprocess.run(_cmd, check=False)
    if _dir_out.exists():
        print(_dir_out.read_text())
elif SKIP_EVAL:
    print('Skipped (SKIP_EVAL=True)')
else:
    print('No checkpoint — skipping direction comparison.')


In [ ]:
# Inspect the RegimeFiLM head: temporal persistence, usage distribution,
# and vol-regime correlation.  Runs only when USE_REGIME_FILM=True.
if not SKIP_EVAL and best_ckpt and USE_REGIME_FILM:
    _hv = '0.1' if EVAL_SMOKE else str(HOURS_VAL)
    _rd_out = eval_out_root / 'regime_diag'
    _rd_out.mkdir(parents=True, exist_ok=True)
    _cmd = [
        sys.executable, '-m', 'finmamba3.eval.regime_diagnostics',
        '--checkpoint', best_ckpt,
        '--config',     config_path_str,
        '--data-val',   val_data_str,
        '--norm-path',  str(norm_path),
        '--out-dir',    str(_rd_out),
        '--hours-val',  _hv,
    ]
    subprocess.run(_cmd, check=False)
    for _p in sorted(_rd_out.glob('regime_diag_*.json')):
        _d = _json.loads(_p.read_text())
        print(f'=== {_p.name} ===')
        print(f"  Temporal persistence : {_d['temporal_persistence']:.3f}  "
              f"[{_d['temporal_persistence_flag']}]")
        print(f"  Usage flag           : {_d['usage_flag']}")
        print(f"  Entropy variability  : {_d['entropy_variability']:.4f}")
        print(f"  Vol-regime |Spearman|: {_d['vol_regime_spearman_abs']:.3f}  "
              f"[{_d['vol_corr_flag']}]")
        print()
elif not SKIP_EVAL and not USE_REGIME_FILM:
    print('Regime diagnostics skipped: USE_REGIME_FILM=False in this run.')
    print('Set USE_REGIME_FILM=True in cell 1 and re-train to enable.')
elif SKIP_EVAL:
    print('Skipped (SKIP_EVAL=True)')
else:
    print('No checkpoint — skipping regime diagnostics.')


In [ ]:
# Market-level volatility regime gap: compare validation metrics on low-vol
# vs high-vol markets.  This is the primary experiment for the RegimeFiLM paper
# claim.  Run on baseline (USE_REGIME_FILM=False) and treatment (True) separately
# and compare the 'gap_out_minus_in' column across the two.
if not SKIP_EVAL and best_ckpt and DATASET == 'polymarket':
    _ht = '0.5' if EVAL_SMOKE else str(HOURS_TRAIN)
    _hv = '0.1' if EVAL_SMOKE else str(HOURS_VAL)
    _gap_out = eval_out_root / 'regime_gap.json'
    _cmd = [
        sys.executable, '-m', 'finmamba3.eval.regime_gap_cli',
        '--checkpoint',     best_ckpt,
        '--config',         config_path_str,
        '--data-train',     train_data_str,
        '--data-val',       val_data_str,
        '--norm-path',      str(norm_path),
        '--split-strategy', 'volatility',
        '--out',            str(_gap_out),
        '--hours-train',    _ht,
        '--hours-val',      _hv,
    ]
    subprocess.run(_cmd, check=False)
    if _gap_out.exists():
        _d = _json.loads(_gap_out.read_text())
        print(f"Split: {_d['split_description']}")
        print(f"In-regime:     {_d['in_regime_markets']} markets, "
              f"{_d['in_regime_ticks']:,} ticks")
        print(f"Out-of-regime: {_d['out_of_regime_markets']} markets, "
              f"{_d['out_of_regime_ticks']:,} ticks")
        print()
        # Show only the most informative metrics.
        _keep = {'reconstruction', 'next_mse', 'direction', 'identity', 'ratio', 'entropy'}
        print(f"{'Metric':<46} {'In-regime':>10} {'Out-regime':>10} {'Gap':>10}")
        print('-' * 80)
        for _row in _d['metrics']:
            if any(_k in _row['metric'] for _k in _keep):
                print(f"{_row['metric']:<46} "
                      f"{_row['in_regime']:>10.4f} "
                      f"{_row['out_of_regime']:>10.4f} "
                      f"{_row['gap_out_minus_in']:>+10.4f}")
elif not SKIP_EVAL and DATASET != 'polymarket':
    print('Regime gap skipped: market-level split requires the Polymarket dataset.')
elif SKIP_EVAL:
    print('Skipped (SKIP_EVAL=True)')
else:
    print('No checkpoint — skipping regime gap.')


In [ ]:
# Linear probing: train a logistic regression on top of frozen posterior latents
# to measure whether the latent encodes vol-regime, imbalance direction, spread
# tier, and next-tick direction.  Requires scikit-learn (installed below if
# INSTALL_SKLEARN=True).  A delta_accuracy < 0.02 on any target flags a blind spot.
if not SKIP_EVAL and best_ckpt:
    if INSTALL_SKLEARN:
        subprocess.check_call(
            [sys.executable, '-m', 'pip', 'install', '-q', 'scikit-learn'],
        )
    _hv = '0.1' if EVAL_SMOKE else str(HOURS_VAL)
    _mw = '64'  if EVAL_SMOKE else '512'
    _probe_out = eval_out_root / 'linear_probe.json'
    _cmd = [
        sys.executable, '-m', 'finmamba3.eval.linear_probe',
        '--checkpoint',   best_ckpt,
        '--config',       config_path_str,
        '--data-val',     val_data_str,
        '--norm-path',    str(norm_path),
        '--out',          str(_probe_out),
        '--hours-val',    _hv,
        '--max-windows',  _mw,
    ]
    subprocess.run(_cmd, check=False)
    if _probe_out.exists():
        _d = _json.loads(_probe_out.read_text())
        print(f"Latents: {_d['n_latents']} vectors x dim {_d['latent_dim']}")
        print(f"\n{'Target':<22} {'Chance':>7} {'Probe':>7} {'Delta':>7}  Note")
        print('-' * 58)
        for _r in _d['results']:
            _flag = 'OK' if _r['delta_accuracy'] > 0.02 else 'LOW (latent blind spot)'
            print(f"{_r['target']:<22} "
                  f"{_r['chance_accuracy']:>7.3f} "
                  f"{_r['probe_accuracy']:>7.3f} "
                  f"{_r['delta_accuracy']:>+7.3f}  {_flag}")
elif SKIP_EVAL:
    print('Skipped (SKIP_EVAL=True)')
else:
    print('No checkpoint — skipping linear probe.')


In [ ]:
# Upload all eval output JSONs to HuggingFace Hub alongside the training logs.
# If HF_TOKEN is not set, outputs are kept locally at eval_outputs/.
_hf_token = get_hf_token()
if not SKIP_EVAL and eval_out_root.exists() and _hf_token and HF_REPO:
    from huggingface_hub import HfApi
    HfApi().upload_folder(
        folder_path=str(eval_out_root),
        path_in_repo=f'eval/{RUN_DATE}',
        repo_id=HF_REPO, repo_type='dataset', token=_hf_token,
    )
    print(f'Eval outputs uploaded to '
          f'https://huggingface.co/datasets/{HF_REPO}/tree/main/eval/{RUN_DATE}')
elif SKIP_EVAL:
    print('Eval skipped (SKIP_EVAL=True) — nothing to upload.')
elif eval_out_root.exists():
    print(f'HF_TOKEN not set — eval outputs saved locally at: {eval_out_root}')
    for _p in sorted(eval_out_root.rglob('*.json')):
        print(f'  {_p.relative_to(eval_out_root)}')
else:
    print('No eval outputs produced — check cell output above for errors.')
